# Evaluate the Fine-Tuned RAFT SLM (Kaggle)

This is an inference notebook for RAFT finetuned SLM, for training notebook please visit: https://www.kaggle.com/code/srikanthmachiraju/raft-finetuning-slm/

This notebook runs on a **Kaggle GPU** instance (T4 / P100 / A10) and:

1. Loads the fine-tuned `llama-3.2-1B-Instruct` adapter from the Hugging Face Hub via Unsloth (4-bit).
2. Loads the held-out RAFT test split (`data/training_data_raft/test.jsonl`).
3. Generates an answer for each record using the **same** chat template / system prompt used at training time.
4. Saves predictions to `slm_predictions.csv`.
5. *(Optional)* Scores predictions with LlamaIndex `Faithfulness`, `Relevancy`, `Correctness` — judge LLM = Azure OpenAI GPT-4o.
6. Writes per-sample scores to `slm_eval_results.csv` and prints a summary.

> The output CSVs flow into **Stage 5** for direct comparison against the GPT-4.1 baseline produced by `llama_rag_evaluate.py`.

## 0. Install dependencies

Mirrors the install block in `raft-finetuning-slm.ipynb` so the same Unsloth / Transformers versions are loaded on the Kaggle GPU. LlamaIndex Azure-OpenAI extras are added for the optional evaluation step.

In [ ]:
%%capture
import os, subprocess
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil   = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"
try:
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except Exception:
    is_t4 = False
_vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq llama-index llama-index-llms-azure-openai pandas tqdm

## 1. Configuration

Update these for your run. Defaults match the model id pushed by the fine-tuning notebook.

In [ ]:
# --- Model & data ---
HF_MODEL_ID    = "sriksmachi/llama32_1bn_instruct_raft"  # fine-tuned model on HF Hub
MAX_SEQ_LEN    = 2048
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.0

# Path to the held-out test split. On Kaggle, attach this repo as a dataset and adjust the path.
# Two common options:
#   1) Add the repo as a Kaggle dataset:  /kaggle/input/<dataset-slug>/data/training_data_raft/test.jsonl
#   2) Upload test.jsonl directly:         /kaggle/input/raft-test/test.jsonl
TEST_JSONL = "/kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/filtered/test.jsonl"

# --- Output paths (Kaggle working dir is persisted as session output) ---
PRED_CSV = "/kaggle/working/slm_predictions.csv"
EVAL_CSV = "/kaggle/working/slm_eval_results.csv"

# --- Optional: cap sample count for a quick smoke run ---
LIMIT = None  # e.g., 20 to test the full pipeline quickly

# --- Run the LlamaIndex evaluation step? ---
RUN_EVAL = True

## 2. Load the fine-tuned model

Unsloth merges the LoRA adapter back into the 4-bit base for fast inference. The same `llama-3.1` chat template used during fine-tuning is applied here.

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = HF_MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Switch to inference mode (Unsloth's 2x faster path)
FastLanguageModel.for_inference(model)
print("Model loaded:", HF_MODEL_ID)

## 3. Load the test split

In [ ]:
import json
from pathlib import Path

def load_jsonl(path, limit=None):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    if limit is not None:
        rows = rows[:limit]
    return rows

records = load_jsonl(TEST_JSONL, LIMIT)
print(f"Loaded {len(records)} test record(s) from {TEST_JSONL}")
print("First record keys:", list(records[0].keys()))

## 4. Generate answers with the fine-tuned SLM

We mirror the **exact** chat-style prompt used during fine-tuning (`raft-finetuning-slm.ipynb`):

- `system`  → the fine-tuning system prompt
- `user`    → `"<Retrieved Documents>: \n{instruction}"` (the `instruction` field already contains the documents + the question)
- `assistant` → generated by the model (CoT + `<ANSWER>:`)

In [ ]:
from tqdm.auto import tqdm

_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using the provided context. "
    "DO NOT use any information that is not included in the <Retrieved Documents>."
    "You should Answer ### Question STRICTLY in this FORMAT: "
    "### Step-by-step reasoning: Use several quotes from <Retrieved Documents>: "
    "##begin_quote## [Relevant text 1] ##end_quote## "
    "##begin_quote## [Relevant text 2] ##end_quote## "
    "Then think step-by-step. <ANSWER>A/B/C/D</ANSWER>"
)

@torch.inference_mode()
def generate_answer(instruction: str) -> str:
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user",   "content": f"<Retrieved Documents>: \n{instruction}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize             = True,
        add_generation_prompt= True,
        return_tensors       = "pt",
    ).to(model.device)
    attention_mask = torch.ones_like(input_ids, device=model.device)

    out = model.generate(
        input_ids       = input_ids,
        attention_mask  = attention_mask,
        max_new_tokens  = MAX_NEW_TOKENS,
        do_sample       = TEMPERATURE > 0,
        temperature     = TEMPERATURE if TEMPERATURE > 0 else 1.0,
        use_cache       = True,
        pad_token_id    = tokenizer.eos_token_id,
    )
    # Slice off the prompt tokens, decode the new tokens only.
    new_tokens = out[0, input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

import pandas as pd

predictions = []
for r in tqdm(records, desc="SLM inference", unit="sample"):
    instr = r.get("instruction") or ""
    if not instr:
        continue
    try:
        ans = generate_answer(instr)
    except Exception as e:
        print("generation failed:", e)
        ans = ""
    predictions.append({
        "id":                r.get("id"),
        "question":          r.get("question", ""),
        "instruction":       instr,
        "reference_answer":  r.get("cot_answer", ""),
        "model_answer":      ans,
        "type": r.get("type")
    })

pred_df = pd.DataFrame(predictions)
Path(PRED_CSV).parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(PRED_CSV, index=False)
print(f"Saved {len(pred_df)} predictions to {PRED_CSV}")
pred_df[["question", "model_answer"]].head(3)

## 6. Done

Outputs in `/kaggle/working/`:

- `slm_predictions.csv` — per-sample model answers (always produced)